# Initial Cohort Time-Series Extraction

**Calling API from extract_timeseries.py**

In [9]:
%load_ext autoreload
%autoreload 2

import duckdb
import pandas as pd
from icu_tft.data.extract_timeseries import build_timeseries, validate_cohort

con = duckdb.connect('../data/mimic.duckdb')

con.execute('''
            CREATE SCHEMA IF NOT EXISTS mimic_hosp;
            CREATE SCHEMA IF NOT EXISTS mimic_icu;
            ''')
print('Schemas created successfully')



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Schemas created successfully


# Map Raw Compressed CSV files to DuckDB Views

No need to copy all datasets into a duckdb file. Views will serve as a much more efficient way of conducting preliminary EDA

In [10]:
con.execute('''
            CREATE OR REPLACE VIEW mimic_hosp.admissions AS SELECT * FROM read_csv_auto('../data/raw/hosp/admissions.csv.gz');
            CREATE OR REPLACE VIEW mimic_hosp.patients AS SELECT * FROM read_csv_auto('../data/raw/hosp/patients.csv.gz');
            CREATE OR REPLACE VIEW mimic_hosp.labevents AS SELECT * FROM read_csv_auto('../data/raw/hosp/labevents.csv.gz');
            ''')

con.execute('''
            CREATE OR REPLACE VIEW mimic_icu.icustays AS SELECT * FROM read_csv_auto('../data/raw/icu/icustays.csv.gz');
            CREATE OR REPLACE VIEW mimic_icu.chartevents AS SELECT * FROM read_csv_auto('../data/raw/icu/chartevents.csv.gz');
            ''')

print('Raw data mapped to duck views successfully')



Raw data mapped to duck views successfully


# Building Initial Cohort Tables



In [11]:
cohort_sql = '''
CREATE TABLE IF NOT EXISTS cohort AS
WITH base_icu AS (
    SELECT 
        ie.subject_id, ie.hadm_id, ie.stay_id, 
        ie.intime AS icu_intime, ie.outtime AS icu_outtime,
        ie.first_careunit, ie.last_careunit,
        EXTRACT(EPOCH FROM (ie.outtime - ie.intime)) / 3600.00 AS icu_los_hours,
        ROW_NUMBER() OVER (
            PARTITION BY ie.hadm_id ORDER BY ie.intime ASC
        ) AS icu_seq_num
    FROM mimic_icu.icustays AS ie
),
first_icu AS (
    SELECT * FROM base_icu WHERE icu_seq_num = 1
),
patient_info AS (
    SELECT 
        p.subject_id, p.gender, p.anchor_age, p.anchor_year, p.dod,
        p.anchor_age + (EXTRACT(YEAR FROM fi.icu_intime)::INT - p.anchor_year) AS age_at_icu_admission,
        fi.stay_id 
    FROM mimic_hosp.patients AS p
    INNER JOIN first_icu AS fi ON p.subject_id = fi.subject_id
),
admission_info AS (
    SELECT 
        a.hadm_id, a.admittime, a.dischtime, a.admission_type, 
        a.insurance, a.language, a.marital_status, a.race, 
        a.hospital_expire_flag, a.deathtime,
        EXTRACT(EPOCH FROM (a.dischtime - a.admittime)) / 3600.0 AS hospital_los_hours
    FROM mimic_hosp.admissions AS a
),
joined AS (
    SELECT 
        fi.subject_id, fi.hadm_id, fi.stay_id, 
        fi.icu_intime, fi.icu_outtime, fi.icu_los_hours, fi.first_careunit,
        pi.gender, pi.anchor_age, pi.age_at_icu_admission, pi.dod,
        ai.admission_type, ai.insurance, ai.marital_status, ai.race, 
        ai.hospital_los_hours, ai.hospital_expire_flag, ai.deathtime
    FROM first_icu AS fi
    INNER JOIN patient_info AS pi ON fi.stay_id = pi.stay_id
    INNER JOIN admission_info AS ai ON fi.hadm_id = ai.hadm_id
),
filtered AS (
    SELECT * FROM joined
    WHERE age_at_icu_admission >= 18
      AND icu_los_hours >= 24.0
      AND NOT (
          deathtime IS NOT NULL
          AND EXTRACT(EPOCH FROM (deathtime - icu_intime)) / 3600.0 < 1.0
      )
      AND NOT (
          deathtime IS NULL
          AND dod = DATE(icu_intime)
          AND icu_los_hours < 1.0
      )
),
labelled AS (
    SELECT 
        subject_id, hadm_id, stay_id, gender, anchor_age, admission_type, 
        first_careunit, icu_los_hours, hospital_los_hours, insurance, race, marital_status,
        CASE 
            WHEN dod IS NOT NULL AND dod <= DATE(icu_intime + INTERVAL '24 hours') THEN 1 
            ELSE 0 
        END AS mortality_24h,
        CAST(hospital_expire_flag AS SMALLINT) AS mortality_inhospital
    FROM filtered
)
SELECT 
    subject_id, hadm_id, stay_id, gender, anchor_age, admission_type, 
    first_careunit, ROUND(icu_los_hours::NUMERIC, 2) AS icu_los_hours, 
    ROUND(hospital_los_hours::NUMERIC, 2) AS hospital_los_hours,
    mortality_24h, mortality_inhospital, insurance, race, marital_status
FROM labelled
ORDER BY subject_id, stay_id;
'''

con.execute(cohort_sql)

print('===' * 50)
print('Cohort Table successfully created and populated')

Cohort Table successfully created and populated


In [12]:
import sys

sys.path.append('../src')

from icu_tft.data.extract_timeseries import build_timeseries, validate_cohort

cohort_df = con.execute('SELECT * FROM cohort').df()

ts = build_timeseries(cohort_df, con)

validate_cohort(ts)

ts.to_parquet('../data/processed/timeseries_features.parquet')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 COHORT TIME-SERIES VALIDATION REPORT
 Total patients (stay_ids) :   67,223
  Patients with complete 24 h  :   67,223  (100.0%)
 Dataset Shape : (1613352, 36)

 Mean Missingness per feature (after foward fill): 
 ----------------------------------------
  dbp                   100.0%  ████████████████████
  potassium             100.0%  ████████████████████
  sodium                100.0%  ████████████████████
  lactate               84.9%  ████████████████
  platelets             79.8%  ███████████████
  creatinine            79.7%  ███████████████
  bicarbonate           78.9%  ███████████████
  bun                   74.1%  ██████████████
  inr                   72.1%  ██████████████
  wbc                   70.4%  ██████████████
  mbp                   67.1%  █████████████
  hemoglobin            56.9%  ███████████
  sbp                   25.7%  █████
  temperature_c         15.5%  ███
  gcs_total             10.0%  ██
  resp_rate              3.6%  
  spo2                   3.3%  
  

# First Cohort Table Population


## Data Shape Overview

+ 67,223 patients overall, 100% of sample have complete 24 hour logs
+ 36 features in total, watch out for this

## Data Missingness Overview

+ Complete missingness for dbp, potassium, sodium after forward fill
+ Majority missing from lactate, platelets, creatinine, bicarbonates, bun, inr, wbc, mbp, hemoglobin
+ Slightly missing: sbp, temperature_c, gcs_total
+ Barely missing: resp_rate, spo2, heart_rate

**Notes**

gcs_total sounds about right to be hovering around 10% as one missing figure from motor, eyes, verbal nulls an entire score. 

# Importing and Using Static_features file

**Calling build_static_features & feature_summary functions**

But first we need to create a view for diagnoses_icd as both functions call for it

In [19]:
con.execute('''
    CREATE OR REPLACE VIEW mimic_hosp.diagnoses_icd 
    AS SELECT * FROM read_csv_auto('../data/raw/hosp/diagnoses_icd.csv.gz');
''')

print('dagnoses_icd view created successfully')

dagnoses_icd view created successfully


In [23]:
from icu_tft.data.static_features import build_static_features, feature_summary

# call both functions from pre-existing cohort_df
static_features_df = build_static_features(cohort_df, con)
feature_summary(static_features_df)

static_features_df.to_parquet('../data/processed/static_features.parquet')
print('\nStatic features successfully process and exported')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

STATIC FEATURES SUMMARY REPORT

Feature: age | Unique : 73 | Missing : 0.0%
 Mean: 63.38 | Std Dev : 16.47
 Min: 18.00 | Median: 65.00 | Max: 91.00

Feature: age_binned | Unique : 4 | Missing : 0.0%
 0 : 9.6%
 1 : 27.3%
 2 : 35.7%
 3 : 27.4%

Feature: gender_M | Unique : 2 | Missing : 0.0%
 0 : 43.5%
 1 : 56.5%

Feature: race_Asian | Unique : 2 | Missing : 0.0%
 0 : 97.0%
 1 : 3.0%

Feature: race_Black | Unique : 2 | Missing : 0.0%
 0 : 89.4%
 1 : 10.6%

Feature: race_Hispanic | Unique : 2 | Missing : 0.0%
 0 : 96.2%
 1 : 3.8%

Feature: race_Other | Unique : 2 | Missing : 0.0%
 0 : 96.3%
 1 : 3.7%

Feature: race_White | Unique : 2 | Missing : 0.0%
 0 : 33.9%
 1 : 66.1%

Feature: admission_type_Elective | Unique : 2 | Missing : 0.0%
 0 : 96.3%
 1 : 3.7%

Feature: admission_type_Emergency | Unique : 2 | Missing : 0.0%
 0 : 20.8%
 1 : 79.2%

Feature: admission_type_Urgent | Unique : 2 | Missing : 0.0%
 0 : 83.1%
 1 : 16.9%

Feature: careunit_Other | Unique : 1 | Missing : 0.0%
 1 : 100.0%